# SDH exp_012 — enrichment 안정화와 하락 클래스 오류 분석

B04와 LR은 고정합니다. support/shrinkage와 class-score 포함 범위만 독립적으로 변경하고, LIHC·DLBC·HNSC·LUSC의 OOF score 분포와 주요 오분류를 함께 분석합니다.

In [ ]:
from pathlib import Path
from time import perf_counter
import gc
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'common').is_dir() and (p / 'experiments').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('프로젝트 루트를 찾지 못했습니다.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.SDH.exp_012_enrichment_stability.preprocessing import (
    B04,
    B04_ID,
    DECLINING_CLASSES,
    EnrichmentCase,
    build_case_matrices,
    make_cases,
    make_context,
)

TRAIN_PATH = PROJECT_ROOT / 'data' / 'raw' / 'train.csv'
RESULTS_DIR = PROJECT_ROOT / 'experiments' / 'SDH' / 'exp_012_enrichment_stability' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
N_SPLITS = 5
SCREEN_SEEDS = (42,)
CONFIRMATION_SEEDS = (42, 52, 62)
B04_EXPECTED_SEED42 = 0.47814168846488037
TARGET_CLASSES = tuple(DECLINING_CLASSES)
print(B04_ID, B04.CONFIG.lr_c, B04.CONFIG.lr_max_iter, TARGET_CLASSES)

## 1. 데이터와 row-local cache

In [ ]:
train = pd.read_csv(TRAIN_PATH, low_memory=False)
genes = [column for column in train.columns if column not in ('ID', 'SUBCLASS')]
labels = train['SUBCLASS'].reset_index(drop=True)
classes = sorted(labels.unique())
context = make_context(train[genes], genes, show_progress=True)
cases = make_cases()
print('train:', train.shape, 'classes:', len(classes))
print('gene-type vocabulary:', context.gene_type_matrix.shape[1])
display(pd.DataFrame([
    {
        'case': case.name,
        'support': case.min_support if case.include_enrichment else None,
        'shrinkage': case.shrinkage if case.include_enrichment else None,
        'excluded_scores': ','.join(case.excluded_scores) or '-',
        'description': case.description,
    }
    for case in cases.values()
]))

## 2. 공통 OOF 평가와 score 수집

모든 validation 행에 대해 예측값, target-class 확률과 표준화된 enrichment score를 저장합니다.

In [ ]:
def evaluate_case(
    case: EnrichmentCase,
    seeds=SCREEN_SEEDS,
    capture_score_classes=TARGET_CLASSES,
):
    per_seed_rows = []
    class_rows = []
    detail_frames = []
    started = perf_counter()

    for seed in seeds:
        splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        prediction = np.empty(len(labels), dtype=object)
        probabilities = {name: np.full(len(labels), np.nan) for name in capture_score_classes}
        enrichment_scores = {name: np.full(len(labels), np.nan) for name in capture_score_classes}
        fold_scores = []
        feature_counts = []
        extra_counts = []
        warning_count = 0

        for fold, (train_index, valid_index) in enumerate(
            splitter.split(np.zeros(len(labels)), labels), start=1
        ):
            fold_started = perf_counter()
            x_train, x_valid, feature_names, metadata = build_case_matrices(
                context, train_index, valid_index, labels, case,
                inner_seed=seed * 100 + fold,
            )
            model = B04.make_model('logistic', seed, B04.CONFIG.lr_max_iter)
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always', ConvergenceWarning)
                model.fit(x_train, labels.iloc[train_index])
            fold_prediction = model.predict(x_valid)
            fold_probability = model.predict_proba(x_valid)
            prediction[valid_index] = fold_prediction
            class_lookup = {name: index for index, name in enumerate(model.classes_)}
            feature_lookup = {name: index for index, name in enumerate(feature_names)}
            for class_name in capture_score_classes:
                probabilities[class_name][valid_index] = (
                    fold_probability[:, class_lookup[class_name]]
                )
                score_name = f'E__gene_type__{class_name}'
                if score_name in feature_lookup:
                    enrichment_scores[class_name][valid_index] = (
                        x_valid[:, feature_lookup[score_name]].toarray().ravel()
                    )
            fold_f1 = f1_score(
                labels.iloc[valid_index], fold_prediction, average='macro', zero_division=0
            )
            fold_scores.append(fold_f1)
            feature_counts.append(len(feature_names))
            extra_counts.append(metadata['extra_feature_count'])
            warning_count += sum(
                issubclass(item.category, ConvergenceWarning) for item in caught
            )
            print(
                f'[{case.name}] seed={seed} fold={fold}/{N_SPLITS} '
                f'f1={fold_f1:.5f} features={len(feature_names):,} '
                f'extra={metadata["extra_feature_count"]} '
                f'time={perf_counter() - fold_started:.1f}s'
            )
            del x_train, x_valid, model
            gc.collect()

        oof_f1 = f1_score(labels, prediction, average='macro', zero_division=0)
        accuracy = accuracy_score(labels, prediction)
        per_class = f1_score(
            labels, prediction, labels=classes, average=None, zero_division=0
        )
        support = labels.value_counts().reindex(classes).to_numpy()
        per_seed_rows.append({
            'case': case.name, 'seed': seed,
            'oof_f1_macro': oof_f1, 'oof_accuracy': accuracy,
            'fold_f1_mean': float(np.mean(fold_scores)),
            'fold_f1_std': float(np.std(fold_scores)),
            'feature_count_min': min(feature_counts),
            'feature_count_max': max(feature_counts),
            'extra_feature_count_min': min(extra_counts),
            'extra_feature_count_max': max(extra_counts),
            'convergence_warning_count': warning_count,
        })
        class_rows.extend({
            'case': case.name, 'seed': seed, 'class': class_name,
            'f1': score, 'support': count
        } for class_name, score, count in zip(classes, per_class, support))
        detail = pd.DataFrame({
            'row': np.arange(len(labels)),
            'ID': train['ID'],
            'true': labels,
            'pred': prediction,
            'correct': labels.to_numpy() == prediction,
            'case': case.name,
            'seed': seed,
        })
        for class_name in capture_score_classes:
            detail[f'score__{class_name}'] = enrichment_scores[class_name]
            detail[f'prob__{class_name}'] = probabilities[class_name]
        detail_frames.append(detail)
        print(f'완료: {case.name} seed={seed} OOF Macro F1={oof_f1:.5f}')

    per_seed = pd.DataFrame(per_seed_rows)
    summary = {
        'case': case.name,
        'min_support': case.min_support if case.include_enrichment else np.nan,
        'shrinkage': case.shrinkage if case.include_enrichment else np.nan,
        'excluded_scores': ','.join(case.excluded_scores),
        'seeds': list(seeds),
        'oof_f1_macro_mean': per_seed['oof_f1_macro'].mean(),
        'oof_f1_macro_std': per_seed['oof_f1_macro'].std(ddof=0),
        'oof_accuracy_mean': per_seed['oof_accuracy'].mean(),
        'feature_count_min': per_seed['feature_count_min'].min(),
        'feature_count_max': per_seed['feature_count_max'].max(),
        'extra_feature_count_min': per_seed['extra_feature_count_min'].min(),
        'extra_feature_count_max': per_seed['extra_feature_count_max'].max(),
        'convergence_warning_count': per_seed['convergence_warning_count'].sum(),
        'elapsed_seconds': perf_counter() - started,
    }
    return {
        'case': case, 'summary': summary, 'per_seed': per_seed,
        'class_f1': pd.DataFrame(class_rows),
        'oof_detail': pd.concat(detail_frames, ignore_index=True),
    }

## 3. seed42 전체 독립 screen

각 case는 별도 셀입니다. 중간에 커널을 끄지 않으면 결과가 `screen_results`에 누적됩니다.

In [ ]:
screen_results = {}
case_name = 'case_00_b04'
screen_results[case_name] = evaluate_case(cases[case_name])
observed_b04 = screen_results[case_name]['summary']['oof_f1_macro_mean']
print('B04 delta vs stored:', observed_b04 - B04_EXPECTED_SEED42)
if abs(observed_b04 - B04_EXPECTED_SEED42) > 0.002:
    raise RuntimeError('B04 재현 차이가 0.002를 넘습니다.')

In [ ]:
case_name = 'case_01_winner_support10_shrink20'
screen_results[case_name] = evaluate_case(cases[case_name])

In [ ]:
case_name = 'case_02_support5'
screen_results[case_name] = evaluate_case(cases[case_name])

In [ ]:
case_name = 'case_03_support20'
screen_results[case_name] = evaluate_case(cases[case_name])

In [ ]:
case_name = 'case_04_shrink10'
screen_results[case_name] = evaluate_case(cases[case_name])

In [ ]:
case_name = 'case_05_shrink50'
screen_results[case_name] = evaluate_case(cases[case_name])

In [ ]:
case_name = 'case_06_exclude_LIHC_score'
screen_results[case_name] = evaluate_case(cases[case_name])

In [ ]:
case_name = 'case_07_exclude_DLBC_score'
screen_results[case_name] = evaluate_case(cases[case_name])

In [ ]:
case_name = 'case_08_exclude_HNSC_score'
screen_results[case_name] = evaluate_case(cases[case_name])

In [ ]:
case_name = 'case_09_exclude_LUSC_score'
screen_results[case_name] = evaluate_case(cases[case_name])

In [ ]:
case_name = 'case_10_exclude_declining4_scores'
screen_results[case_name] = evaluate_case(cases[case_name])

In [ ]:
leaderboard = (
    pd.DataFrame([result['summary'] for result in screen_results.values()])
    .sort_values('oof_f1_macro_mean', ascending=False)
    .reset_index(drop=True)
)
b04_score = leaderboard.loc[
    leaderboard['case'].eq('case_00_b04'), 'oof_f1_macro_mean'
].iloc[0]
winner_score = leaderboard.loc[
    leaderboard['case'].eq('case_01_winner_support10_shrink20'),
    'oof_f1_macro_mean',
].iloc[0]
leaderboard['delta_vs_b04'] = leaderboard['oof_f1_macro_mean'] - b04_score
leaderboard['delta_vs_exp011_winner'] = (
    leaderboard['oof_f1_macro_mean'] - winner_score
)
leaderboard.to_csv(RESULTS_DIR / 'leaderboard_seed42.csv', index=False)
pd.concat([result['class_f1'] for result in screen_results.values()]).to_csv(
    RESULTS_DIR / 'class_f1_seed42.csv', index=False
)
display(leaderboard)

## 4. 하락 클래스 score 분포와 주요 오분류 분석

In [ ]:
def make_target_error_pairs(result, targets=TARGET_CLASSES):
    details = result['oof_detail'].query('seed == 42')
    rows = []
    for target in targets:
        target_rows = details.loc[details['true'].eq(target)]
        errors = target_rows.loc[~target_rows['pred'].eq(target)]
        counts = errors['pred'].value_counts()
        for rank, (predicted, count) in enumerate(counts.head(5).items(), start=1):
            rows.append({
                'case': result['case'].name, 'true_class': target,
                'rank': rank, 'predicted_as': predicted, 'count': count,
                'error_share_within_true': count / len(target_rows),
            })
    return pd.DataFrame(rows)

error_pairs = pd.concat([
    make_target_error_pairs(screen_results['case_00_b04']),
    make_target_error_pairs(screen_results['case_01_winner_support10_shrink20']),
], ignore_index=True)
error_pairs.to_csv(RESULTS_DIR / 'target_error_pairs_seed42.csv', index=False)
display(error_pairs)

In [ ]:
winner_detail = screen_results[
    'case_01_winner_support10_shrink20'
]['oof_detail'].query('seed == 42').copy()
distribution_rows = []
by_true_rows = []
for target in TARGET_CLASSES:
    score_column = f'score__{target}'
    probability_column = f'prob__{target}'
    winner_detail['segment'] = np.select(
        [
            winner_detail['true'].eq(target) & winner_detail['pred'].eq(target),
            winner_detail['true'].eq(target) & ~winner_detail['pred'].eq(target),
            ~winner_detail['true'].eq(target) & winner_detail['pred'].eq(target),
        ],
        ['target_correct', 'target_missed', 'false_positive'],
        default='other',
    )
    for segment, group in winner_detail.groupby('segment'):
        distribution_rows.append({
            'target_score': target, 'segment': segment, 'n': len(group),
            'score_mean': group[score_column].mean(),
            'score_std': group[score_column].std(),
            'score_q10': group[score_column].quantile(0.10),
            'score_q25': group[score_column].quantile(0.25),
            'score_median': group[score_column].median(),
            'score_q75': group[score_column].quantile(0.75),
            'score_q90': group[score_column].quantile(0.90),
            'probability_mean': group[probability_column].mean(),
            'probability_median': group[probability_column].median(),
        })
    for true_class, group in winner_detail.groupby('true'):
        by_true_rows.append({
            'target_score': target, 'true_class': true_class, 'n': len(group),
            'score_mean': group[score_column].mean(),
            'score_median': group[score_column].median(),
            'probability_mean': group[probability_column].mean(),
        })

score_distribution = pd.DataFrame(distribution_rows)
score_by_true = pd.DataFrame(by_true_rows).sort_values(
    ['target_score', 'score_mean'], ascending=[True, False]
)
score_distribution.to_csv(
    RESULTS_DIR / 'target_score_distribution_seed42.csv', index=False
)
score_by_true.to_csv(
    RESULTS_DIR / 'target_score_by_true_class_seed42.csv', index=False
)
display(score_distribution)
display(score_by_true.groupby('target_score').head(6))

In [ ]:
class_f1 = pd.concat([
    screen_results['case_00_b04']['class_f1'],
    screen_results['case_01_winner_support10_shrink20']['class_f1'],
])
class_comparison = class_f1.pivot_table(
    index=['class', 'support'], columns='case', values='f1'
).reset_index()
class_comparison['delta'] = (
    class_comparison['case_01_winner_support10_shrink20']
    - class_comparison['case_00_b04']
)
display(class_comparison.loc[class_comparison['class'].isin(TARGET_CLASSES)])

## 5. B04와 상위 비기준 2개 3-seed 확인

In [ ]:
top_nonbaseline = leaderboard.loc[
    leaderboard['case'] != 'case_00_b04', 'case'
].head(2).tolist()
confirmation_names = ['case_00_b04', *top_nonbaseline]
print('3-seed confirmation:', confirmation_names)
confirmation_results = {
    name: evaluate_case(
        cases[name], seeds=CONFIRMATION_SEEDS, capture_score_classes=()
    )
    for name in confirmation_names
}

In [ ]:
confirmation_leaderboard = (
    pd.DataFrame([result['summary'] for result in confirmation_results.values()])
    .sort_values('oof_f1_macro_mean', ascending=False)
    .reset_index(drop=True)
)
confirmation_b04 = confirmation_leaderboard.loc[
    confirmation_leaderboard['case'].eq('case_00_b04'),
    'oof_f1_macro_mean',
].iloc[0]
confirmation_leaderboard['delta_vs_b04'] = (
    confirmation_leaderboard['oof_f1_macro_mean'] - confirmation_b04
)
confirmation_leaderboard.to_csv(
    RESULTS_DIR / 'leaderboard_confirmation.csv', index=False
)
pd.concat([result['per_seed'] for result in confirmation_results.values()]).to_csv(
    RESULTS_DIR / 'per_seed_confirmation.csv', index=False
)
pd.concat([result['class_f1'] for result in confirmation_results.values()]).to_csv(
    RESULTS_DIR / 'class_f1_confirmation.csv', index=False
)
display(confirmation_leaderboard)

## 6. exp12 champion 제출 파일 생성

아래 셀은 `case_04_shrink10`을 전체 train으로 학습해 test를 예측한다. train/test 원본을 합치지 않으며, exact-event와 gene×event-type vocabulary는 train에서만 만든다. 전체 train의 enrichment 입력도 내부 5-fold OOF로 생성한다. 새 커널에서도 이 섹션의 코드 셀만 위에서 아래로 실행할 수 있다.

In [ ]:
from pathlib import Path
import gc
import json
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from sklearn.exceptions import ConvergenceWarning

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'common').is_dir() and (p / 'experiments').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('프로젝트 루트를 찾지 못했습니다.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.SDH.exp_012_enrichment_stability.preprocessing import (
    B04,
    _b04_builder,
    _cross_fitted_scores,
    _standardize_and_filter,
    make_cases,
    make_submission_context,
)

TRAIN_PATH = PROJECT_ROOT / 'data' / 'raw' / 'train.csv'
TEST_PATH = PROJECT_ROOT / 'data' / 'raw' / 'test.csv'
SAMPLE_PATH = PROJECT_ROOT / 'data' / 'raw' / 'sample_submission.csv'
OUTPUT_DIR = (
    PROJECT_ROOT
    / 'experiments'
    / 'SDH'
    / 'exp_012_enrichment_stability'
    / 'results'
)
SUBMISSION_PATH = (
    OUTPUT_DIR
    / 'submission_exp012_b04_gene_type_shrink10_seed42.csv'
)
SUBMISSION_METADATA_PATH = SUBMISSION_PATH.with_name(
    SUBMISSION_PATH.stem + '_metadata.json'
)
SUBMISSION_SEED = 42
SUBMISSION_CASE_NAME = 'case_04_shrink10'

train = pd.read_csv(TRAIN_PATH, low_memory=False)
test = pd.read_csv(TEST_PATH, low_memory=False)
sample_submission = pd.read_csv(SAMPLE_PATH)
genes = [column for column in train.columns if column not in ('ID', 'SUBCLASS')]
labels = train['SUBCLASS'].reset_index(drop=True)
classes = sorted(labels.unique())
cases = make_cases()
assert list(train.columns) == ['ID', 'SUBCLASS', *genes]
assert list(test.columns) == ['ID', *genes]
assert list(sample_submission.columns) == ['ID', 'SUBCLASS']
assert sample_submission['ID'].tolist() == test['ID'].tolist()
assert int(train[genes].isna().sum().sum()) == 0
assert B04.normalise_cell(np.nan) == ()
assert B04.normalise_cell('WT') == ()
print('test:', test.shape)
print('test 결측 셀 수(참고용, 판정에 사용하지 않음):', int(test[genes].isna().sum().sum()))
print('submission:', SUBMISSION_PATH)

In [ ]:
submission_context, train_only_context, submission_audit = (
    make_submission_context(
        train[genes],
        test[genes],
        genes,
        show_progress=True,
    )
)
full_train_index = np.arange(len(train))
test_index = np.arange(len(train), len(train) + len(test))
assert submission_context.gene_type_matrix.shape[0] == len(train) + len(test)
assert train_only_context.gene_type_matrix.shape[0] == len(train)
display(pd.DataFrame([submission_audit]))

In [ ]:
# B04도 train에서 확정한 vocabulary와 선택 규칙만 test에 적용한다.
submission_b04_builder = _b04_builder(submission_context)
x_b04_train, x_b04_test, b04_features = submission_b04_builder.build(
    full_train_index, test_index, labels
)

# test 행을 전혀 포함하지 않은 train-only cache로 다시 만들어 완전 일치를 검사한다.
train_only_b04_builder = _b04_builder(train_only_context)
x_train_only_check, _, train_only_features = train_only_b04_builder.build(
    full_train_index, full_train_index, labels
)
base_leakage_check = bool(
    b04_features == train_only_features
    and (x_b04_train != x_train_only_check).nnz == 0
)
assert base_leakage_check, 'test 행이 B04 train 설계행렬에 영향을 줬습니다.'
print('B04 leakage check:', base_leakage_check)
print('B04 features:', len(b04_features))
del x_train_only_check, train_only_b04_builder
gc.collect()

In [ ]:
# 전체 train 입력은 내부 5-fold OOF, test 입력은 전체 train weight 적용값이다.
submission_case = cases[SUBMISSION_CASE_NAME]
train_scores, test_scores, enrichment_features = _cross_fitted_scores(
    submission_context,
    full_train_index,
    test_index,
    labels,
    submission_case,
    inner_seed=SUBMISSION_SEED,
)
train_scores, test_scores, enrichment_features = _standardize_and_filter(
    train_scores, test_scores, enrichment_features
)
assert len(enrichment_features) == 26
assert np.isfinite(train_scores).all()
assert np.isfinite(test_scores).all()

x_full_train = sparse.hstack(
    [x_b04_train, sparse.csr_matrix(train_scores)], format='csr'
)
x_test = sparse.hstack(
    [x_b04_test, sparse.csr_matrix(test_scores)], format='csr'
)
submission_features = [*b04_features, *enrichment_features]
assert x_full_train.shape == (len(train), len(submission_features))
assert x_test.shape == (len(test), len(submission_features))
print('enrichment features:', len(enrichment_features))
print('total features:', len(submission_features))

In [ ]:
submission_model = B04.make_model(
    'logistic', SUBMISSION_SEED, B04.CONFIG.lr_max_iter
)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', ConvergenceWarning)
    submission_model.fit(x_full_train, labels)
submission_prediction = submission_model.predict(x_test)
submission_warning_count = sum(
    issubclass(item.category, ConvergenceWarning) for item in caught
)
assert submission_warning_count == 0
assert set(submission_prediction).issubset(set(classes))
print('prediction complete:', len(submission_prediction))
print('convergence warnings:', submission_warning_count)

In [ ]:
submission = sample_submission.copy()
submission['SUBCLASS'] = submission_prediction
assert len(submission) == len(test)
assert submission['ID'].tolist() == test['ID'].tolist()
assert int(submission.isna().sum().sum()) == 0

submission_metadata = {
    'experiment_id': 'SDH-exp012',
    'case': submission_case.name,
    'model': 'logistic_lbfgs',
    'seed': SUBMISSION_SEED,
    'lr_c': B04.CONFIG.lr_c,
    'lr_max_iter': B04.CONFIG.lr_max_iter,
    'class_weight': 'balanced',
    'min_support': submission_case.min_support,
    'shrinkage': submission_case.shrinkage,
    'base_feature_count': len(b04_features),
    'enrichment_feature_count': len(enrichment_features),
    'total_feature_count': len(submission_features),
    'convergence_warning_count': submission_warning_count,
    'base_leakage_check': base_leakage_check,
    **submission_audit,
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission.to_csv(SUBMISSION_PATH, index=False)
SUBMISSION_METADATA_PATH.write_text(
    json.dumps(submission_metadata, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
display(submission.head())
display(submission['SUBCLASS'].value_counts().sort_index().to_frame('count'))
display(pd.DataFrame([submission_metadata]))
print('제출 파일 저장 완료:', SUBMISSION_PATH)
print('메타데이터 저장 완료:', SUBMISSION_METADATA_PATH)